In [19]:
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

## Data Loading

In [20]:
df = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")

## Feature Engineering

In [21]:
# ⚙️ Import Transformer
from sklearn.preprocessing import FunctionTransformer

# 🗑️ Drop Columns
drop_cols = ["Id", "Alley", "PoolQC", "Fence", "MiscFeature"]


# ------------------------------
# 🏗️ Feature Engineering Function
# ------------------------------
def added_features(df):
    df_out = df.copy()

    # Drop high null columns
    df_out.drop(columns=drop_cols, errors="ignore")
    
    # Age
    df_out["PropertyAge"] = df_out["YrSold"] - df_out["YearBuilt"]

    # Area
    df_out["TotalSF"] = df_out["TotalBsmtSF"] + df_out["1stFlrSF"] + df_out["2ndFlrSF"]
    df_out["TotalBath"] = df_out["FullBath"] + 0.5 * df_out["HalfBath"] + df_out["BsmtFullBath"] + 0.5 * df_out["BsmtHalfBath"]

    # Remodeled? 
    df_out["HasRemodeled"] = (df_out["YearRemodAdd"] != df_out["YearBuilt"]).astype(object)

    # Binary features for property amenities
    df_out["HasPool"] = (df_out["PoolArea"] > 0).astype(int)
    df_out["HasBasement"] = (df_out["TotalBsmtSF"] > 0).astype(int)
    df_out["HasFireplace"] = (df_out["Fireplaces"] > 0).astype(int)
    df_out["Has2ndFloor"] = (df_out["2ndFlrSF"] > 0).astype(object)
    df_out["HasGarage"] = (df_out["GarageArea"] > 0).astype(object)

    # Convert some numeric features to categorical
    df_out["YrSold_cat"] = df_out["YrSold"].astype(object)
    df_out["MoSold_cat"] = df_out["MoSold"].astype(object)
    df_out["YearBuilt_cat"] = df_out["YearBuilt"].astype(object)
    df_out["MSSubClass_cat"] = df_out["MSSubClass"].astype(object)

    # Good Overall Quality
    df_out["OverallGrade"] = df_out["OverallQual"] * df_out["OverallCond"]
    df_out["QualityArea"] = df_out["OverallQual"] * df_out["GrLivArea"]
    
    
    return df_out

# ------------------------------
# 🔄 Wrap Feature Engineering into Transformer
# ------------------------------
feature_transformer = FunctionTransformer(added_features)

## Preprocessing Pipelines

In [22]:
new_cols_categorical = pd.Index(["HasRemodeled", "Has2ndFloor", "HasGarage",
                                 "YrSold_cat", "MoSold_cat", "YearBuilt_cat",
                                 "MSSubClass_cat"])
new_cols_numeric = pd.Index(["PropertyAge", "TotalSF", "TotalBath", 
                             "HasFireplace", "HasBasement", "HasPool",
                             "OverallGrade", "QualityArea"])


categorical_columns = df.select_dtypes(include=["object", "category"]).columns.append(new_cols_categorical)
numerical_columns = df.select_dtypes(include=["int64", "float64"]).columns.append(new_cols_numeric)


# 🎯 Drop Target Column from Numerics
numerical_columns = numerical_columns.drop("SalePrice")

In [23]:
# ⚙️ Import Sklearn Tools for Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 🗃️ Categorical Pipeline
cat_transform = Pipeline(steps = [
                                  ("imputer", SimpleImputer(strategy="constant", fill_value="None")),
                                  ("onehot", OneHotEncoder(handle_unknown="ignore", sparse = False))
                                 ])

# 🔢 Numerical Pipeline
num_transform = Pipeline(steps = [
                                  ("imputer", SimpleImputer(strategy="mean")),
                                  ("scaler", StandardScaler())
                                 ])

In [24]:
preprocessor = ColumnTransformer(transformers=[("cat", cat_transform, categorical_columns),
                                               ("num", num_transform, numerical_columns)
                                               ], 
                                                 remainder = "passthrough")

# ------------------------------
# 🏗️ Create Full Pipeline with Feature Engineering
# ------------------------------
pipeline = Pipeline(steps=[
    ("engineer", feature_transformer),
    ("preprocessor", preprocessor)])

# 📝 Apply Pipeline to Dataset
X = df.drop("SalePrice", axis=1)
y = np.log(df["SalePrice"]) #normalize dependent variable 

X_preprocessed = pipeline.fit_transform(X)

test_preprocessed = pipeline.transform(test)

In [25]:
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge, ElasticNet, ElasticNetCV
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import mean_squared_error
import optuna

## Hyperparameter Tuning (Optuna)

In [26]:
# 🎯 Define Objectives for Optuna
objectives = {}

# ------------------------------
# 🌲 LightGBM Objective (Sane Params)
# ------------------------------
def objective_lgbm(trial):
    params = {
        # 1. Boosting Settings
        "n_estimators": 2000, # Increased to allow lower LRs to converge
        "verbose": -1,
        "random_state": 42,
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        
        # 2. Tree Structure
        # num_leaves should be < 2^(max_depth) to avoid overfitting
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "num_leaves": trial.suggest_int("num_leaves", 20, 100), 
        
        # 3. Regularization (Crucial)
        # min_child_samples is the most important parameter to stop overfitting
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 0.5),
        
        # 4. Sampling
        # subsample_freq MUST be > 0 for subsample to work in LGBM
        "subsample": trial.suggest_float("subsample", 0.5, 0.95),
        "subsample_freq": trial.suggest_int("subsample_freq", 1, 5),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.95),
        
        # 5. L1/L2 Regularization
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-5, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-5, 10.0, log=True),
    }
    
    model = LGBMRegressor(**params)
    
    # Using a slight negative score for minimization
    score = -cross_val_score(model, X_preprocessed, y, 
                             cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1).mean()
    return score

objectives["LightGBM"] = objective_lgbm


# ------------------------------
# 🏛️ Ridge Objective (Sane Params)
# ------------------------------
def objective_ridge(trial):
    # 1. Alpha (Regularization Strength)
    # Widen the net significantly. 
    # - 1e-5 approaches LinearRegression (OLS).
    # - 1e5 is heavy regularization (useful if features are highly collinear).
    alpha = trial.suggest_float("alpha", 1e-5, 20000.0, log=True)
    
    # 2. Solver
    # Usually 'auto' is fine, but sometimes it helps to let Optuna 
    # pick between 'svd' (stable) and 'lsqr'/'sag' (fast/iterative).
    # However, for pure accuracy on mid-sized data, 'auto' handles this well.
    
    # 3. Tolerance
    # Only used if solver='sag' or 'lsqr'. 
    # Tighter tolerance can squeeze out slight performance gains.
    tol = trial.suggest_float("tol", 1e-5, 1e-3, log=True)

    model = Ridge(alpha=alpha, tol=tol, random_state=42)
    
    score = -cross_val_score(model, X_preprocessed, y,
                             cv=5, scoring="neg_root_mean_squared_error").mean()
    return score

objectives["Ridge"] = objective_ridge

# ------------------------------
# ⚡ ElasticNet Objective (Sane Params)
# ------------------------------
def objective_elastic(trial):
    # 1. Alpha (Penalty Strength)
    # Like Ridge, we need a wider range. 
    # 100.0 covers high-noise scenarios. 1e-5 allows near-OLS behavior.
    alpha = trial.suggest_float("alpha", 1e-5, 100.0, log=True)
    
    # 2. L1 Ratio (Mixing Parameter)
    # 0.0 = Ridge, 1.0 = Lasso. 
    # Sklearn warns against using exactly 0.0 with ElasticNet (use Ridge class instead).
    # We search from 0.01 to 1.0 to safely cover the spectrum.
    l1_ratio = trial.suggest_float("l1_ratio", 0.01, 1.0)
    
    # 3. Convergence Settings
    # 'random' selection is often faster than 'cyclic' for large feature sets
    # tol controls optimization precision; tuning it balances speed vs accuracy.
    tol = trial.suggest_float("tol", 1e-5, 1e-2, log=True)
    selection = trial.suggest_categorical("selection", ["cyclic", "random"])

    model = ElasticNet(
        alpha=alpha, 
        l1_ratio=l1_ratio,
        tol=tol,
        selection=selection,
        random_state=42, 
        max_iter=10000 # High max_iter is required for small alphas
    )
    
    score = -cross_val_score(model, X_preprocessed, y,
                             cv=5, scoring="neg_root_mean_squared_error").mean()
    return score

objectives["ElasticNet"] = objective_elastic



# ------------------------------
# 🎢 Kernel Ridge Regression Objective (Sane Params)
# ------------------------------
def objective_krr(trial):
    # 1. Alpha
    # KRR often benefits from very slight regularization if the kernel fits well.
    alpha = trial.suggest_float("alpha", 1e-4, 10.0, log=True)
    
    # 2. Kernel Selection
    # I REMOVED 'linear'. You already have a standard Ridge regression in your stack.
    # KRR with a linear kernel is mathematically identical to Ridge.
    # 'laplacian' is added: it's like RBF but sharper (exp(-|x|)) vs (exp(-x^2)).
    kernel = trial.suggest_categorical("kernel", ["polynomial", "rbf", "laplacian"])
    
    # 3. Gamma (Crucial!)
    # Your original code missed this. For RBF/Poly/Laplacian, gamma controls
    # how "far" the influence of a single training example reaches.
    # Without tuning gamma, KRR is essentially guessing the bandwidth.
    gamma = trial.suggest_float("gamma", 1e-4, 5.0, log=True)
    
    # 4. Conditional Logic
    if kernel == "polynomial":
        # Keep degree low. Degree > 3 tends to oscillate wildly at boundaries.
        degree = trial.suggest_int("degree", 2, 3)
        coef0 = trial.suggest_float("coef0", 0.1, 10.0)
        model = KernelRidge(alpha=alpha, kernel=kernel, gamma=gamma, degree=degree, coef0=coef0)
    else:
        # RBF and Laplacian utilize gamma and alpha
        model = KernelRidge(alpha=alpha, kernel=kernel, gamma=gamma)
        
    score = -cross_val_score(model, X_preprocessed, y,
                             cv=5, scoring="neg_root_mean_squared_error").mean()
    return score

objectives["KernelRidge"] = objective_krr



# ------------------------------
# 🐉 XGBoost Objective (Sane Params)
# ------------------------------
def objective_xgb(trial):
    params = {
        "objective": "reg:squarederror",
        "tree_method": "hist",
        "random_state": 42,
        "n_estimators": 2000, # Increased to allow convergence at lower learning rates
        
        # 1. Learning Rate & Depth
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        
        # 2. Conservative Splitting (CRITICAL)
        # min_child_weight: The sum of instance weight needed in a child. 
        # Higher values prevent the model from learning specific noise samples.
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        
        # gamma: Minimum loss reduction required to make a further partition.
        # A non-zero gamma makes the algorithm conservative (prunes the tree).
        "gamma": trial.suggest_float("gamma", 1e-4, 1.0, log=True),
        
        # 3. Stochastic Sampling
        # Avoid 1.0 to maintain diversity (row/column subsampling)
        "subsample": trial.suggest_float("subsample", 0.5, 0.95),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.95),
        
        # 4. Regularization
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
    }
    
    model = XGBRegressor(**params)
    
    score = -cross_val_score(model, X_preprocessed, y,
                             cv=5, scoring="neg_root_mean_squared_error").mean()
    return score

objectives["XGBoost"] = objective_xgb

from sklearn.linear_model import Lasso
from sklearn.svm import SVR

# ------------------------------
# 🤠 Lasso Objective
# ------------------------------
def objective_lasso(trial):
    # Lasso benefits from high max_iter when alpha is small to ensure convergence
    alpha = trial.suggest_float("alpha", 1e-4, 10.0, log=True)
    model = Lasso(alpha=alpha, random_state=42, max_iter=10000)
    
    score = -cross_val_score(model, X_preprocessed, y, 
                             cv=5, scoring="neg_root_mean_squared_error").mean()
    return score
objectives["Lasso"] = objective_lasso

# # ------------------------------
# # 🔮 SVR Objective
# # ------------------------------
# def objective_svr(trial):
#     # SVR is sensitive to scaling; ensure X_preprocessed is scaled (e.g., StandardScaler)
#     C = trial.suggest_float("C", 1e-3, 100.0, log=True)
#     epsilon = trial.suggest_float("epsilon", 1e-4, 1.0, log=True)
#     kernel = trial.suggest_categorical("kernel", ["linear", "poly", "rbf"])
    
#     # Conditional hyperparameters based on kernel
#     if kernel == "poly":
#         degree = trial.suggest_int("degree", 2, 4)
#         gamma = trial.suggest_categorical("gamma", ["scale", "auto"])
#         coef0 = trial.suggest_float("coef0", 0.0, 5.0)
#         model = SVR(C=C, epsilon=epsilon, kernel=kernel, degree=degree, gamma=gamma, coef0=coef0)
#     elif kernel == "rbf":
#         gamma = trial.suggest_categorical("gamma", ["scale", "auto"])
#         model = SVR(C=C, epsilon=epsilon, kernel=kernel, gamma=gamma)
#     else:
#         model = SVR(C=C, epsilon=epsilon, kernel=kernel)

#     score = -cross_val_score(model, X_preprocessed, y, 
#                              cv=5, scoring="neg_root_mean_squared_error").mean()
#     return score
# objectives["SVR"] = objective_svr

from sklearn.ensemble import ExtraTreesRegressor

# ------------------------------
# 🌳 Extra Trees Objective
# ------------------------------
def objective_et(trial):
    params = {
        "n_estimators": 1000,
        "n_jobs": -1,
        "random_state": 42,
        "max_depth": trial.suggest_int("max_depth", 3, 25),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 15),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_float("max_features", 0.5, 1.0),
        "bootstrap": trial.suggest_categorical("bootstrap", [True, False])
    }
    model = ExtraTreesRegressor(**params)
    score = -cross_val_score(model, X_preprocessed, y, 
                             cv=5, scoring="neg_root_mean_squared_error").mean()
    return score
objectives["ExtraTrees"] = objective_et

from catboost import CatBoostRegressor

# ------------------------------
# 🐱 CatBoost Objective
# ------------------------------
def objective_catboost(trial):
    params = {
        "iterations": 1000,
        "random_seed": 42,
        "verbose": 0,
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "border_count": trial.suggest_int("border_count", 32, 255),
    }
    # Note: CatBoost creates its own internal DMatrix, so straight fitting is usually fine
    # but standard sklearn CV wrapper is used here for consistency.
    model = CatBoostRegressor(**params)
    score = -cross_val_score(model, X_preprocessed, y, 
                             cv=5, scoring="neg_root_mean_squared_error").mean()
    return score
objectives["CatBoost"] = objective_catboost

In [27]:
# # ------------------------------
# # 🏁 Run Optuna Hyperparameter Optimization
# # ------------------------------
# best_params = {}
# best_scores = {}

# for name, objective in objectives.items():
#     print(f"\n🔎 Optimizing {name}...")
#     study = optuna.create_study(direction="minimize")
#     study.optimize(objective, n_trials=5, show_progress_bar=True) 
#     # Best params were found with 150 n_trials.
    
#     best_params[name] = study.best_params
#     best_scores[name] = study.best_value
    
#     print(f"✅ {name} best RMSE: {study.best_value:.5f}")
#     print(f"Best params: {study.best_params}")

In [28]:
import optuna
import pandas as pd

# 1. Setup
optuna.logging.set_verbosity(optuna.logging.WARNING) # Suppress huge logs
results = {} # To store best parameters
model_performance = {} # To store best CV scores

print(f"🚀 Starting optimization for {len(objectives)} models...")
print("-" * 50)

# 2. Optimization Loop
for model_name, objective_func in objectives.items():
    print(f"⏳ Optimizing {model_name}...")
    
    # Create study
    study = optuna.create_study(direction="minimize")
    study.optimize(objective_func, n_trials=50, show_progress_bar=True)
    
    # Store results
    results[model_name] = study.best_params
    model_performance[model_name] = -study.best_value # Convert back to positive RMSE
    
    print(f"✅ {model_name} Done! Best RMSE: {-study.best_value:.5f}")

# 3. Show Leaderboard
leaderboard = pd.DataFrame(list(model_performance.items()), columns=["Model", "RMSE"])
leaderboard = leaderboard.sort_values("RMSE")
print("\n🏆 Optimization Leaderboard:")
print(leaderboard)

🚀 Starting optimization for 8 models...
--------------------------------------------------
⏳ Optimizing LightGBM...


  0%|          | 0/50 [00:00<?, ?it/s]

✅ LightGBM Done! Best RMSE: -0.12553
⏳ Optimizing Ridge...


  0%|          | 0/50 [00:00<?, ?it/s]

✅ Ridge Done! Best RMSE: -0.13694
⏳ Optimizing ElasticNet...


  0%|          | 0/50 [00:00<?, ?it/s]

... (8 outputs left out)

✅ ExtraTrees Done! Best RMSE: -0.13040
⏳ Optimizing CatBoost...


  0%|          | 0/50 [00:00<?, ?it/s]

[W 2025-11-17 16:35:04,538] Trial 3 failed with parameters: {'learning_rate': 0.0029852655656382693, 'depth': 10, 'l2_leaf_reg': 8.061366292734457, 'bagging_temperature': 0.32988218303534544, 'border_count': 236} because of the following error: KeyboardInterrupt('').
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/optuna/study/_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_172/3280683711.py", line 274, in objective_catboost
    score = -cross_val_score(model, X_preprocessed, y,
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py", line 515, in cross_val_score
    cv_results = cross_validate(
                 ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py", line 266, in cross_validate
    results = parallel(
              ^^^^^^

KeyboardInterrupt: 

In [33]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import RidgeCV

# Re-instantiate models with the Best Params found + Static Params
# We use the **results['Name'] syntax to unpack the optimized args

lgbm_final = LGBMRegressor(
    n_estimators=1000, 
    random_state=42, 
    verbose=-1, 
    **results["LightGBM"]
)

xgb_final = XGBRegressor(
    n_estimators=1000, 
    objective="reg:squarederror", 
    tree_method="hist", 
    random_state=42, 
    **results["XGBoost"]
)

ridge_final = Ridge(random_state=42, **results["Ridge"])

elastic_final = ElasticNet(random_state=42, max_iter=20000, **results["ElasticNet"])

lasso_final = Lasso(random_state=42, max_iter=10000, **results["Lasso"])

# Logic for KernelRidge (handling the conditional kernel parameters)
# We need to check if 'degree' or 'coef0' exists in the best params
krr_params = results["KernelRidge"]
krr_final = KernelRidge(**krr_params)

# Logic for SVR (handling conditional params)
# svr_final = SVR(**results["SVR"])

# ExtraTrees (if you added it)
et_final = ExtraTreesRegressor(
    n_estimators=1000, 
    n_jobs=-1, 
    random_state=42, 
    **results["ExtraTrees"]
)

# CatBoost (if you added it)
# cat_final = CatBoostRegressor(
#     iterations=1000, 
#     verbose=0, 
#     random_seed=42, 
#     **results["CatBoost"]
# )

# ------------------------------
# 🏗️ Build the Stacking Regressor
# ------------------------------

# Define the base estimators
estimators = [
    ('lgbm', lgbm_final),
    ('xgb', xgb_final),
    ('ridge', ridge_final),
    ('elastic', elastic_final),
    ('lasso', lasso_final),
    ('krr', krr_final),
    # ('svr', svr_final),
    ('et', et_final),
    # ('cat', cat_final)
]

# Define the Meta-Learner
# RidgeCV is a standard, robust choice for the meta-learner to avoid overfitting
stacking_model = StackingRegressor(
    estimators=estimators,
    final_estimator=RidgeCV(), 
    cv=5,
    n_jobs=-1,
    passthrough=False # Set to True if you want original features fed to the meta-learner
)

print("🤖 Stacking Regressor is ready to fit!")

🤖 Stacking Regressor is ready to fit!


In [34]:
stacking_model.fit(X_preprocessed, y)

StackingRegressor(cv=5,
                  estimators=[('lgbm',
                               LGBMRegressor(colsample_bytree=0.5030948814182918,
                                             learning_rate=0.04893239114102978,
                                             max_depth=10,
                                             min_split_gain=0.05444237080423042,
                                             n_estimators=1000, num_leaves=93,
                                             random_state=42,
                                             reg_alpha=0.04517314006335207,
                                             reg_lambda=5.093027550417711e-05,
                                             subsample=0.5524301688808525,
                                             subsample_freq=2, verbose=-1)),
                              ('xgb...
                                          tol=0.0006425693541822782)),
                              ('lasso',
                               Lasso(alpha=0.0005177778050034228,
                                     max_iter=10000, random_state=42)),
                              ('krr',
                               KernelRidge(alpha=0.01854629657583775,
                                           gamma=0.0004275999164508341,
                                           kernel='laplacian')),
                              ('et',
                               ExtraTreesRegressor(max_depth=21,
                                                   max_features=0.9752142325539571,
                                                   n_estimators=1000, n_jobs=-1,
                                                   random_state=42))],
                  final_estimator=RidgeCV(), n_jobs=-1)

In [31]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_predict

# 1. Gather Predictions
# We use the 'estimators' list defined in the previous step
prediction_df = pd.DataFrame()
prediction_df['True Label'] = y # Optional: to see corr with actual target

print("📊 Generating out-of-fold predictions for correlation analysis...")
print("-" * 50)

for name, model in estimators:
    print(f"   ... generating predictions for: {name}")
    
    # cross_val_predict generates predictions for each data point 
    # when it was in the validation set.
    preds = cross_val_predict(model, X_preprocessed, y, cv=5, n_jobs=-1)
    prediction_df[name] = preds

# 2. Calculate Correlation Matrix
corr_matrix = prediction_df.drop(columns=['True Label']).corr()

# 3. Plot Heatmap
plt.figure(figsize=(12, 10))
sns.set_style("white")

# Mask the upper triangle (since it's symmetrical)
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix, 
    annot=True, 
    fmt=".3f", 
    cmap="coolwarm", 
    mask=mask,
    vmin=0.8, # Adjust min to see subtle differences (models usually corr > 0.8)
    vmax=1.0,
    linewidths=0.5,
    square=True,
    cbar_kws={"shrink": .75}
)

plt.title("Base Model Prediction Correlation Matrix", fontsize=16)
plt.show()

📊 Generating out-of-fold predictions for correlation analysis...
--------------------------------------------------
   ... generating predictions for: lgbm
   ... generating predictions for: xgb
   ... generating predictions for: ridge
   ... generating predictions for: elastic
   ... generating predictions for: lasso
   ... generating predictions for: krr
   ... generating predictions for: et


KeyboardInterrupt: 

In [35]:
# 2. Predict on Test Data
# Ensure 'X_test_preprocessed' is your processed test set matching X_preprocessed columns
print("🔮 Generating predictions...")
final_predictions = stacking_model.predict(test_preprocessed)
final_pred = np.expm1(final_predictions)

# Optional: Clip predictions if your target has known bounds 
# (e.g., if y can't be negative, uncomment below)
# final_predictions = np.clip(final_predictions, a_min=0, a_max=None)

# 3. Create Submission DataFrame
# Replace 'test_df' with the dataframe containing your original test IDs
submission = pd.DataFrame({
    'Id': test['Id'],  # Adjust column name if it's 'id' or 'ID'
    'SalePrice': final_pred # Adjust target column name as required
})

# 4. Export to CSV
submission_filename = "submission_stacking_v1.csv"
submission.to_csv(submission_filename, index=False)

print(f"🚀 Submission saved to: {submission_filename}")
print(submission.head())

🔮 Generating predictions...
🚀 Submission saved to: submission_stacking_v1.csv
     Id      SalePrice
0  1461  121105.148405
1  1462  152480.030640
2  1463  179488.930939
3  1464  193536.267003
4  1465  192068.813517


In [37]:
X_preprocessed.to_csv('preprocessed.csv')

AttributeError: 'numpy.ndarray' object has no attribute 'to_csv'